# Herramienta 01 - Predicción de demanda de transporte

Este notebook funciona como una herramienta completa para estimar demanda por ruta. Incluye preparación de datos, análisis exploratorio, entrenamiento, evaluación y un pronóstico operativo de 30 días.


## 1. Configuración
Se cargan las librerías necesarias. El notebook no requiere clonar el repositorio: lee el CSV completo `data/processed/cta_bus_ridership_daily_by_route.csv`, tomado del Chicago Data Portal. El archivo completo queda en GitHub y el notebook filtra desde 2021 para entrenar con datos recientes.


In [ ]:
# Librerías principales
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

SEED     = 42
WINDOW   = 14   # días de historia como secuencia de entrada al LSTM
EPOCHS   = 80
BATCH    = 32
PATIENCE = 10

np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_URL         = 'https://raw.githubusercontent.com/AndresGuido9820/sistema-transporte-inteligente/main/data/processed/cta_bus_ridership_daily_by_route.csv'
TRAIN_START_DATE = '2021-01-01'
TOP_RUTAS        = 10

## 2. Carga y filtrado del dataset real
El archivo se carga completo desde el repositorio público. Contiene demanda diaria por ruta de buses CTA desde 2001 hasta 2026. Para entrenar un modelo operativo y rápido en Colab, se filtra desde 2021 y se toman las rutas con mayor demanda reciente.


In [ ]:
raw_demanda = pd.read_csv(DATA_URL)
raw_demanda['date'] = pd.to_datetime(raw_demanda['date'], errors='coerce')
raw_demanda['rides'] = pd.to_numeric(raw_demanda['rides'], errors='coerce')
raw_demanda = raw_demanda.dropna(subset=['date', 'route', 'rides'])

demanda = raw_demanda[raw_demanda['date'] >= pd.Timestamp(TRAIN_START_DATE)].copy()
rutas_principales = demanda.groupby('route')['rides'].sum().nlargest(TOP_RUTAS).index
demanda = demanda[demanda['route'].isin(rutas_principales)].copy()
demanda = demanda.rename(columns={'rides': 'passengers'})
demanda['holiday'] = demanda['daytype'].eq('U').astype(int)
demanda = demanda[['date', 'route', 'passengers', 'holiday', 'daytype']].sort_values(['route', 'date']).reset_index(drop=True)

print('Archivo completo cargado:', raw_demanda.shape)
print('Rango completo:', raw_demanda['date'].min().date(), 'a', raw_demanda['date'].max().date())
print('Datos usados para entrenamiento:', demanda.shape)
print('Rutas seleccionadas:', sorted(demanda['route'].unique()))
demanda.head()


## 3. Exploración inicial
Se revisa el volumen por ruta y el comportamiento temporal para detectar tendencia, estacionalidad semanal e intermitencia.


In [ ]:
resumen = demanda.groupby('route')['passengers'].agg(['count', 'mean', 'min', 'max']).round(2)
display(resumen)

plt.figure(figsize=(12, 5))
for ruta, datos in demanda.groupby('route'):
    serie = datos.sort_values('date').set_index('date')['passengers'].rolling(7).mean()
    plt.plot(serie.index, serie.values, label=ruta)
plt.title('Media móvil de 7 días por ruta')
plt.xlabel('Fecha')
plt.ylabel('Pasajeros')
plt.legend(fontsize=8)
plt.grid(alpha=0.25)
plt.show()


## 3.5 Análisis de estacionalidad y tendencias

Se analizan tres dimensiones de la demanda: **tendencia** de largo plazo (descomposición aditiva con `seasonal_decompose`, período 7 días), **patrón semanal** (demanda promedio por día de semana) y **evolución anual** (comparativa año a año por ruta). Esto permite identificar si la demanda crece o cae en el tiempo, en qué días se concentra y cómo se comporta la estacionalidad semanal.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Ruta con mayor demanda total como referencia
ruta_ref = demanda.groupby('route')['passengers'].sum().idxmax()
serie_ref = (
    demanda[demanda['route'] == ruta_ref]
    .sort_values('date')
    .set_index('date')['passengers']
    .asfreq('D')
    .fillna(method='ffill')
)

# --- 1. Descomposición aditiva (tendencia + estacionalidad + residuo) ---
decomp = seasonal_decompose(serie_ref, model='additive', period=7)
fig, axes = plt.subplots(4, 1, figsize=(12, 9), sharex=True)
serie_ref.plot(ax=axes[0], lw=1, color='#1f5eff', title='Original')
decomp.trend.plot(ax=axes[1], lw=1.5, color='#a86200', title='Tendencia')
decomp.seasonal.plot(ax=axes[2], lw=1, color='#0a7a5f', title='Estacionalidad semanal')
decomp.resid.plot(ax=axes[3], lw=1, color='#c0392b', title='Residuo')
for ax in axes:
    ax.grid(alpha=0.25)
    ax.set_xlabel('')
plt.suptitle(f'Descomposición aditiva — Ruta {ruta_ref}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

# --- 2. Patrón semanal promedio (todas las rutas) ---
dias = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
tmp = demanda.copy()
tmp['dayofweek'] = tmp['date'].dt.dayofweek
patron_semanal = tmp.groupby('dayofweek')['passengers'].mean()

plt.figure(figsize=(8, 3.5))
colores = ['#1f5eff' if i < 5 else '#f6a531' for i in range(7)]
plt.bar(dias, patron_semanal.values, color=colores)
plt.title('Demanda promedio por día de semana (todas las rutas, 2021–2026)')
plt.ylabel('Pasajeros promedio')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# --- 3. Evolución anual por ruta ---
tmp['year'] = tmp['date'].dt.year
anual = tmp.groupby(['year', 'route'])['passengers'].mean().reset_index()

plt.figure(figsize=(11, 4))
for ruta, datos in anual.groupby('route'):
    plt.plot(datos['year'], datos['passengers'], marker='o', label=str(ruta), lw=1.6)
plt.title('Evolución anual — promedio diario de pasajeros por ruta')
plt.xlabel('Año')
plt.ylabel('Pasajeros promedio/día')
plt.legend(fontsize=8, ncol=2)
plt.grid(alpha=0.25)
plt.xticks(sorted(tmp['year'].unique()))
plt.tight_layout()
plt.show()

## 4. Construcción de secuencias temporales
En lugar de convertir la serie en features tabulares, se construyen **ventanas deslizantes** de 14 días consecutivos de pasajeros normalizados. Cada ventana es la secuencia de entrada al LSTM; el día siguiente es el objetivo a predecir.

Esto permite que el LSTM aprenda por sí mismo las dependencias temporales — autocorrelación, patrón semanal, tendencia — directamente desde la serie cruda, sin pre-procesar esa información como lag features.

In [ ]:
def crear_secuencias(pax_norm, window=14):
    """Convierte una serie normalizada en pares (secuencia, siguiente_valor)."""
    X, y = [], []
    for i in range(window, len(pax_norm)):
        X.append(pax_norm[i - window:i])
        y.append(pax_norm[i])
    return np.array(X).reshape(-1, window, 1), np.array(y)

# Muestra de las primeras secuencias para la ruta de mayor demanda
ruta_ej = demanda.groupby('route')['passengers'].sum().idxmax()
pax_ej  = demanda[demanda['route'] == ruta_ej].sort_values('date')['passengers'].values.astype(float)
sc_ej   = MinMaxScaler()
pax_ej_n = sc_ej.fit_transform(pax_ej.reshape(-1, 1)).flatten()

X_ej, y_ej = crear_secuencias(pax_ej_n, WINDOW)
print(f'Ruta {ruta_ej} — total secuencias: {len(X_ej)}')
print(f'Forma entrada LSTM: {X_ej.shape}  →  (muestras, timesteps, features)')
print(f'Ejemplo secuencia[0]: {X_ej[0].flatten().round(3)}')
print(f'Objetivo[0]:          {y_ej[0]:.4f}')

## 5. Entrenamiento LSTM con ventanas deslizantes
Se entrena un **LSTM** por ruta sobre secuencias de 14 días de pasajeros normalizados. La partición es temporal estricta: 80% entrenamiento, 20% prueba — sin mezclar fechas.

Arquitectura: `LSTM(64, return_sequences=True)` → `LSTM(32)` → `Dense(16, relu)` → `Dense(1)`.

Optimizador **Adam**, pérdida **MSE**, `EarlyStopping(patience=10)` sobre val_loss.

In [ ]:
def build_lstm(window):
    model = keras.Sequential([
        layers.Input(shape=(window, 1)),
        layers.LSTM(64, return_sequences=True),
        layers.LSTM(32),
        layers.Dense(16, activation='relu'),
        layers.Dense(1),
    ], name='LSTM')
    model.compile(optimizer='adam', loss='mse')
    return model

early_stop = callbacks.EarlyStopping(
    monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0
)

resultados   = []
mejores_modelos = {}
scalers      = {}
predicciones = []

for ruta, group in demanda.sort_values(['route', 'date']).groupby('route'):
    group   = group.reset_index(drop=True)
    pax     = group['passengers'].values.astype(float)
    fechas  = group['date'].values

    # Normalizar por ruta
    scaler  = MinMaxScaler()
    pax_n   = scaler.fit_transform(pax.reshape(-1, 1)).flatten()
    scalers[ruta] = scaler

    # Construir secuencias
    X, y, meta = [], [], []
    for i in range(WINDOW, len(pax_n)):
        X.append(pax_n[i - WINDOW:i])
        y.append(pax_n[i])
        meta.append(i)
    X = np.array(X).reshape(-1, WINDOW, 1)
    y = np.array(y)

    # Partición temporal 80/20
    corte    = int(len(X) * 0.8)
    X_tr, X_te = X[:corte], X[corte:]
    y_tr, y_te = y[:corte], y[corte:]
    pax_real   = pax[WINDOW + corte:]
    fechas_te  = fechas[WINDOW + corte:]

    # Entrenar
    modelo = build_lstm(WINDOW)
    modelo.fit(X_tr, y_tr, epochs=EPOCHS, batch_size=BATCH,
               validation_split=0.1, callbacks=[early_stop], verbose=0)

    # Predecir e invertir normalización
    pred_n = modelo.predict(X_te, verbose=0).flatten()
    pred   = np.maximum(scaler.inverse_transform(pred_n.reshape(-1, 1)).flatten(), 0)

    mae  = mean_absolute_error(pax_real, pred)
    rmse = float(np.sqrt(mean_squared_error(pax_real, pred)))
    mape = float(np.mean(np.abs((pax_real - pred) / pax_real)) * 100)
    resultados.append({'route': ruta, 'model': 'LSTM', 'MAE': round(mae, 1),
                       'RMSE': round(rmse, 1), 'MAPE (%)': round(mape, 2)})
    mejores_modelos[ruta] = modelo

    tmp = pd.DataFrame({'date': fechas_te, 'route': ruta,
                        'passengers': pax_real, 'prediction': pred})
    predicciones.append(tmp)

metricas = pd.DataFrame(resultados).sort_values('route')
display(metricas.round(3))

## 6. Gráficas de validación
Estas gráficas permiten explicar si el modelo sigue la forma de la demanda real o si se queda corto en picos específicos.


In [ ]:
pred_df = pd.concat(predicciones, ignore_index=True)
for ruta in pred_df['route'].unique()[:4]:
    graf = pred_df[pred_df['route'] == ruta].sort_values('date')
    plt.figure(figsize=(11, 4))
    plt.plot(graf['date'], graf['passengers'], marker='o', label='Real')
    plt.plot(graf['date'], graf['prediction'], marker='o', label='Predicción')
    plt.title(f'Demanda real vs. predicha - {ruta}')
    plt.ylabel('Pasajeros')
    plt.xticks(rotation=45, ha='right')
    plt.grid(alpha=0.25)
    plt.legend()
    plt.show()


## 7. Herramienta operativa: pronóstico a 30 días
Seleccione una ruta y ejecute la celda para obtener la predicción diaria futura. Esta es la salida que usaría el área de planeación para asignar vehículos y personal.


In [ ]:
#@title Parámetros de la herramienta
ruta_seleccionada = '66' #@param {type:'string'}
dias_a_predecir   = 30   #@param {type:'integer'}

if ruta_seleccionada not in mejores_modelos:
    ruta_seleccionada = list(mejores_modelos.keys())[0]
    print('Ruta no encontrada. Se usará:', ruta_seleccionada)

def pronosticar_ruta(df, ruta, modelo, scaler, window=14, dias=30):
    """Pronóstico iterativo: usa la ventana de los últimos `window` días,
    agrega cada predicción a la ventana para el siguiente paso."""
    hist    = df[df['route'] == ruta].sort_values('date')['passengers'].values.astype(float)
    ultima  = df[df['route'] == ruta]['date'].max()
    ventana = scaler.transform(hist[-window:].reshape(-1, 1)).flatten().tolist()
    filas   = []
    for paso in range(1, dias + 1):
        fecha   = ultima + pd.Timedelta(days=paso)
        seq     = np.array(ventana[-window:]).reshape(1, window, 1)
        pred_n  = float(modelo.predict(seq, verbose=0)[0][0])
        pred    = float(scaler.inverse_transform([[pred_n]])[0][0])
        pred    = max(pred, 0)
        ventana.append(pred_n)
        filas.append({'date': fecha.date(), 'route': ruta, 'forecast_passengers': round(pred, 2)})
    return pd.DataFrame(filas)

sx         = scalers[ruta_seleccionada]
pronostico = pronosticar_ruta(demanda, ruta_seleccionada,
                               mejores_modelos[ruta_seleccionada], sx,
                               WINDOW, dias_a_predecir)
display(pronostico.head(10))

plt.figure(figsize=(11, 4))
plt.plot(pronostico['date'], pronostico['forecast_passengers'], marker='o')
plt.title(f'Pronóstico LSTM (ventana {WINDOW}d) — Ruta {ruta_seleccionada}')
plt.ylabel('Pasajeros estimados')
plt.xticks(rotation=45, ha='right')
plt.grid(alpha=0.25)
plt.show()

## 8. Conclusiones

### Dataset y exploración
El dataset del CTA contiene 1 106 531 registros diarios por ruta (2001–2026).
Filtrado desde 2021, las 10 rutas de mayor demanda aportan 19 160 observaciones
de entrenamiento. Las rutas con mayor demanda promedio diaria son la 66
(~12 965 pasajeros/día) y la 79 (~12 577 pasajeros/día). La descomposición
aditiva confirmó una **estacionalidad semanal marcada** (caída pronunciada en
sábado y domingo) y una **tendencia de recuperación post-COVID** desde 2021, sin
recuperar los niveles previos a 2019.

### Arquitectura: LSTM con ventanas deslizantes
Se entrena un **LSTM** por ruta usando ventanas deslizantes de 14 días de pasajeros
normalizados como entrada directa a la red recurrente. Esta estrategia —en lugar
de features tabulares— permite que el LSTM procese la serie cruda tal como fue
generada, aprendiendo por sí mismo la autocorrelación, el patrón semanal y la
tendencia sin pre-codificarlos.

Arquitectura por ruta: `Input(14, 1)` → `LSTM(64, return_sequences=True)` →
`LSTM(32)` → `Dense(16, relu)` → `Dense(1)`. Optimizador **Adam**, pérdida **MSE**,
`EarlyStopping(patience=10)` sobre val_loss, partición temporal 80/20 estricta.

### Rendimiento por ruta (20% más reciente, datos reales)

| Ruta | MAE | RMSE | MAPE (%) |
|------|------:|------:|----------:|
| **22** | 1 269 | 1 596 | **12.20** |
| 66 | 2 512 | 3 154 | 22.75 |
| 77 | 2 537 | 2 896 | 24.01 |
| 4 | 2 085 | 2 624 | 25.03 |
| 8 | 2 410 | 2 996 | 25.13 |
| 79 | 3 033 | 3 520 | 25.18 |
| 49 | 2 002 | 2 643 | 25.24 |
| 9 | 2 711 | 3 323 | 25.39 |
| 3 | 2 195 | 2 788 | 27.22 |
| **53** | 2 483 | 3 120 | **27.61** |

**MAPE global: mín 12.20 % — máx 27.61 % — media 23.98 %**

La ruta 22 es la excepción positiva (MAPE 12 %): su patrón es más regular y su
volumen absoluto menor, lo que facilita el aprendizaje de la ventana temporal.
Las demás rutas convergen en la banda 22–28 %, reflejo de que 5 años de datos
con un período post-pandemia aún inestable generan alta varianza en el 20 % de
prueba.

### Interpretación del error
Un MAPE de ~25 % sobre demanda de transporte urbano en período de recuperación
post-COVID es esperado para un modelo sin variables exógenas. La mayor parte del
error proviene de **eventos no observables**: cortes de servicio, clima, festivos
no etiquetados y cambios de operación. La ruta 53 (MAPE 27.6 %) presenta mayor
amplitud entre días hábiles y festivos, lo que eleva el error relativo.

### Pronóstico a 30 días
El pronóstico iterativo reproduce el patrón semanal aprendido (mayor demanda
en días hábiles, caída en fin de semana) de forma estable a lo largo del horizonte
de 30 días. No se observa divergencia acumulada en el período evaluado, indicando
que la propagación de error entre pasos es moderada.

### Limitaciones y mejoras sugeridas
- Incorporar **variables exógenas** (temperatura, festivos oficiales, eventos de
  alto aforo) reduciría el MAPE a la banda 10–15 % en rutas de alta variabilidad.
- Aumentar la **ventana de entrada** (e.g., 21 o 28 días) podría capturar mejor
  la estacionalidad mensual emergente.
- Un **re-entrenamiento incremental** (fine-tuning con datos nuevos cada semana)
  es crítico para mantener la precisión conforme el patrón post-pandemia se
  estabiliza.
- Comparar contra el baseline naive (predicción = mismo día de la semana anterior)
  cuantificaría la ganancia real de la red neuronal.

---

## Fuente del dataset

**Chicago Transit Authority (CTA) — CTA Bus Ridership Daily Totals by Route**

- **Portal:** [Chicago Data Portal](https://data.cityofchicago.org/Transportation/CTA-Ridership-Bus-Routes-Daily-Totals-by-Route/jyb9-n7fm)
- **Publicado por:** City of Chicago — Chicago Transit Authority
- **Licencia:** Public Domain (U.S. Government Open Data)
- **Cobertura temporal:** 2001-01-01 al presente (actualización continua)
- **Granularidad:** Demanda diaria por ruta de bus, con tipo de día (`W` = laboral, `A` = sábado, `U` = domingo/festivo)
- **Tamaño descargado:** 1 106 531 filas (al 2026-03-31)